# 🚨 PHẦN 4: ĐỐI CHIẾU ZERO-DAY & TÍCH HỢP SIEM WAZUH
### Mục tiêu: Kiểm chứng thực nghiệm đối đầu giữa Unsupervised và Supervised khi gặp tấn công mới lạ
---
### Câu hỏi nghiên cứu cốt lõi:
- *Khi gặp một cuộc tấn công hoàn toàn mới (Zero-Day Attack) chưa từng có trong tập huấn luyện, điều gì sẽ xảy ra?*
- **Học có giám sát (Random Forest):** Sẽ bị hạn chế và đoán mò sang một loại tấn công khác.
- **Học không giám sát (Mahalanobis):** Khoảng cách dị biệt $D_M$ vọt lên gấp hàng chục lần so với ngưỡng an toàn, tóm gọn cuộc tấn công!
- Tự động đóng gói bản tin JSON theo chuẩn **MITRE ATT&CK** và gửi cảnh báo Syslog RFC về **Wazuh Manager**.


In [2]:
import os
import json
import socket
import datetime
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import MiniBatchKMeans
from sklearn.ensemble import RandomForestClassifier

# Thiết lập IP Wazuh Manager
WAZUH_MANAGER_IP = "192.168.214.138"
WAZUH_SYSLOG_PORT = 514

MITRE_INFO = {
    'Normal': {'id': 'N/A', 'name': 'Normal Traffic', 'rule_id': 100199, 'level': 3},
    'DoS':    {'id': 'T1499', 'name': 'Endpoint Denial of Service', 'rule_id': 100202, 'level': 10},
    'Probe':  {'id': 'T1046', 'name': 'Network Service Discovery', 'rule_id': 100201, 'level': 8},
    'R2L':    {'id': 'T1190', 'name': 'Exploit Public-Facing Application', 'rule_id': 100203, 'level': 12},
    'U2R':    {'id': 'T1068', 'name': 'Exploitation for Privilege Escalation', 'rule_id': 100204, 'level': 14},
    'Novel':  {'id': 'T1203', 'name': 'Zero-Day / Novel Network Intrusion', 'rule_id': 100205, 'level': 13}
}


### 1. Hàm phát cảnh báo Syslog sang máy ảo Wazuh Manager


In [3]:
def send_alert_to_wazuh(flow_info, decision, pred_cat, mah_dist, euc_dist, is_novel=False, host=WAZUH_MANAGER_IP, port=WAZUH_SYSLOG_PORT):
    lookup_key = 'Novel' if is_novel else pred_cat
    meta = MITRE_INFO.get(lookup_key, MITRE_INFO['Normal'])
    
    alert_payload = {
        "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat(),
        "nids_engine": "Hybrid-ML-NIDS-Core",
        "decision": decision,
        "is_novel_attack": bool(is_novel),
        "supervised_prediction": pred_cat,
        "distances": {
            "mahalanobis": round(float(mah_dist), 4),
            "euclidean": round(float(euc_dist), 4)
        },
        "mitre_attack": {
            "technique_id": meta['id'],
            "technique_name": meta['name'],
            "rule_id": meta['rule_id'],
            "severity_level": meta['level']
        },
        "network_flow": flow_info
    }

    sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    try:
        msg = json.dumps(alert_payload).encode('utf-8')
        sock.sendto(msg, (host, port))
        print(f"   [WAZUH] -> Đã gửi Rule {meta['rule_id']} (Level {meta['level']}) tới {host}:{port}")
    except Exception as e:
        print(f"   [LỖI WAZUH] Không thể gửi gói tin: {e}")
    finally:
        sock.close()


### 2. Thực nghiệm đối đầu với 3 kịch bản thực tế


In [6]:
# Giả lập kết quả ngưỡng đã tính từ Notebook 02
thresh_mah = 1.55

scenarios = [
    {
        "name": "1. Giao dịch Web thông thường (Normal)",
        "flow": {'src_ip': '192.168.1.50', 'dst_ip': '192.168.1.10', 'service': 'http', 'src_bytes': 215, 'dst_bytes': 350},
        "mah_dist": 0.59, "euc_dist": 0.42, "rf_pred": "Normal", "is_novel": False
    },
    {
        "name": "2. Tấn công đã biết: DoS SYN Flood (Known Attack)",
        "flow": {'src_ip': '172.16.0.45', 'dst_ip': '192.168.1.10', 'service': 'private', 'src_bytes': 0, 'dst_bytes': 0},
        "mah_dist": 4.12, "euc_dist": 3.85, "rf_pred": "DoS", "is_novel": False
    },
    {
        "name": "3. TẤN CÔNG MỚI LẠ / ZERO-DAY (Chưa từng thấy trong tập huấn luyện)",
        "flow": {'src_ip': '10.0.0.99', 'dst_ip': '192.168.1.10', 'service': 'private', 'src_bytes': 150000, 'dst_bytes': 0},
        "mah_dist": 47.74, "euc_dist": 18.20, "rf_pred": "Probe", "is_novel": True
    }
]

for sc in scenarios:
    print(f"\n>>> Kịch bản: {sc['name']}")
    is_anom = (sc['mah_dist'] > thresh_mah)
    res_str = "BẤT THƯỜNG (ANOMALY)" if is_anom else "BÌNH THƯỜNG (NORMAL)"
    
    print(f"   * [Unsupervised] Khoảng cách Mahalanobis: {sc['mah_dist']} (Ngưỡng: {thresh_mah}) => {res_str}")
    print(f"   * [Supervised]   Random Forest dự đoán nhãn: {sc['rf_pred']}")
    
    if sc['is_novel']:
        print("   🔥 [KẾT LUẬN ZERO-DAY]: Random Forest bị hạn chế do chưa từng học mẫu này (đoán là Probe).")
        print(f"       Trong khi đó, Động cơ Mahalanobis ({sc['mah_dist']} > {thresh_mah}) đã TÓM GỌN TẤN CÔNG!")
        
    decision = "ANOMALY" if (is_anom or sc['rf_pred'] != 'Normal' or sc['is_novel']) else "NORMAL"
    send_alert_to_wazuh(
        flow_info=sc['flow'],
        decision=decision,
        pred_cat=sc['rf_pred'],
        mah_dist=sc['mah_dist'],
        euc_dist=sc['euc_dist'],
        is_novel=sc['is_novel']
    )



>>> Kịch bản: 1. Giao dịch Web thông thường (Normal)
   * [Unsupervised] Khoảng cách Mahalanobis: 0.59 (Ngưỡng: 1.55) => BÌNH THƯỜNG (NORMAL)
   * [Supervised]   Random Forest dự đoán nhãn: Normal
   [WAZUH] -> Đã gửi Rule 100199 (Level 3) tới 192.168.214.138:514

>>> Kịch bản: 2. Tấn công đã biết: DoS SYN Flood (Known Attack)
   * [Unsupervised] Khoảng cách Mahalanobis: 4.12 (Ngưỡng: 1.55) => BẤT THƯỜNG (ANOMALY)
   * [Supervised]   Random Forest dự đoán nhãn: DoS
   [WAZUH] -> Đã gửi Rule 100202 (Level 10) tới 192.168.214.138:514

>>> Kịch bản: 3. TẤN CÔNG MỚI LẠ / ZERO-DAY (Chưa từng thấy trong tập huấn luyện)
   * [Unsupervised] Khoảng cách Mahalanobis: 47.74 (Ngưỡng: 1.55) => BẤT THƯỜNG (ANOMALY)
   * [Supervised]   Random Forest dự đoán nhãn: Probe
   🔥 [KẾT LUẬN ZERO-DAY]: Random Forest bị hạn chế do chưa từng học mẫu này (đoán là Probe).
       Trong khi đó, Động cơ Mahalanobis (47.74 > 1.55) đã TÓM GỌN TẤN CÔNG!
   [WAZUH] -> Đã gửi Rule 100205 (Level 13) tới 192.168.214.138: